#Handling Incremental load

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/consolidated_pipeline/1_setup/utils

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","orders","Data Source")


In [0]:
catalog=dbutils.widgets.get('catalog')
data_source=dbutils.widgets.get('data_source')

base_path=f"s3://sportsbar-aws-dp/orders/"
landing_path=f"{base_path}/landing/"
processed_path=f"{base_path}/processed/"

bronze_table=f"{catalog}.{bronze_schema}.{data_source}"
silver_table=f"{catalog}.{silver_schema}.{data_source}"
gold_table=f"{catalog}.{gold_schema}.sb_{data_source}"

In [0]:
bronze_table,silver_table,gold_table

#Bronze Processing

In [0]:
from pyspark.sql import functions as F

# Corrected syntax: Place .option() BEFORE calling .csv()
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{landing_path}/*.csv") \
    .withColumn("read_timestamp", F.current_timestamp()) \
    .select("*", "_metadata.file_name", "_metadata.file_size")

df=df.withColumn("order_qty",F.col("order_qty").cast("double"))

#Append to existing bronze orders file

In [0]:
df.write.format("delta").mode("append").option("mergeSchema","true").option("delta.enableChangeDataFeed","true").saveAsTable(bronze_table)

#Staging table to process only new records

In [0]:
df.write.format("delta").mode("overwrite").option("delta.enableChangeDataFeed","true")\
    .saveAsTable(f"{catalog}.{bronze_schema}.staging_{data_source}")

In [0]:
files=dbutils.fs.ls(landing_path)
for file_info in files:
    dbutils.fs.mv(file_info.path,f"{processed_path}/{file_info.name}",True)

#Silver Processing

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.staging_{data_source};")


In [0]:

df_orders=df_bronze.where(F.col("order_qty").isNotNull())
df_orders = df_orders.withColumn(
    "customer_id",
    F.when(
        F.col("customer_id").isNotNull() & 
        F.col("customer_id").cast("string").rlike("^[0-9]+$"), 
        F.col("customer_id").cast("string")
    ).otherwise("99999")
)

In [0]:
df_orders=df_orders.withColumn("order_placement_date",F.regexp_replace(F.col("order_placement_date"),r"[A-Za-z]+,\s*",""))
df_orders.show(10)

In [0]:
df_orders=df_orders.withColumn("order_placement_date",F.coalesce(F.try_to_date(F.col("order_placement_date"),"yyyy/MM/dd"),\
                                                       F.try_to_date(F.col("order_placement_date"),"dd-MM-yyyy"),\
                                                       F.try_to_date(F.col("order_placement_date"),"dd/MM/yyyy"),\
                                                       F.try_to_date(F.col("order_placement_date"),"MMMM dd, yyyy")

                                                    ))

In [0]:
df_orders=df_orders.dropDuplicates(["order_id","order_placement_date","customer_id","product_id","order_qty"])

In [0]:
df_orders=df_orders.withColumn("product_id",F.col("product_id").cast("string"))


In [0]:
df_orders.agg(F.max("order_placement_date"),F.min("order_placement_date")).show()

In [0]:
df_products=spark.table("fmcg.silver.products")
df_joined=df_orders.join(df_products.select("product_code","product_id"),on="product_id",how="inner")

df_joined.select("customer_id").distinct().show()

In [0]:
if not(spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option("delta.enableChangeDataFeed","true").mode("overwrite")\
        .option("mergeSchema","true").saveAsTable(f"{catalog}.{silver_schema}.{data_source}")
else:
    silver_delta=DeltaTable.forName(spark,silver_table)
    silver_delta.alias("silver").merge(source=df_joined.alias("bronze"),condition="silver.order_placement_date = bronze.order_placement_date AND silver.order_id=bronze.order_id AND silver.product_code=bronze.product_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
df_joined.write.format("delta").option("delta.enableChangeDataFeed","true").mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.staging_{data_source}")

#Gold Processing

In [0]:
df_gold=spark.sql(f"""select order_id, order_placement_date as date, customer_id as customer_code,
product_code, product_id, order_qty as sold_quantity from {catalog}.{silver_schema}.staging_{data_source}""")
df_gold.show(2)

In [0]:
df_gold.orderBy(F.col("date").desc()).show(10)

In [0]:
from delta.tables import DeltaTable

# Ensure your gold_table variable is mapped out cleanly at the top
# gold_table = f"{catalog}.{gold_schema}.{data_source}"

# 1. Check if the Gold table exists in the metastore catalog
if not spark.catalog.tableExists(gold_table):
    print(f"Table {gold_table} does not exist. Initializing fresh Gold Delta Table...")
    df_gold.write \
        .format("delta") \
        .option("delta.enableChangeDataFeed", "true") \
        .option("mergeSchema", "true") \
        .mode("overwrite") \
        .saveAsTable(gold_table)
else:
    print(f"Table {gold_table} found. Executing incremental Gold Delta Merge...")
    gold_delta = DeltaTable.forName(spark, gold_table)
    
    # 2. Execute the merge using accurate and clear aliases ("gold" and "source")
    gold_delta.alias("gold").merge(
        source=df_gold.alias("source"),
        condition="""
            gold.date = source.date 
            AND gold.order_id = source.order_id 
            AND gold.product_code = source.product_code 
            AND gold.customer_code = source.customer_code
        """
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()


#Merging with parent company

In [0]:
df_child=spark.sql(f"select order_placement_date as date from {catalog}.{silver_schema}.staging_{data_source}")
incremental_month=df_child.select(F.trunc("date","MM").alias("start_month")).distinct()
incremental_month.createOrReplaceTempView("incremental_view")


In [0]:
%sql
select date,product_code,customer_code,sold_quantity from fmcg.gold.sb_orders

In [0]:
monthly_table=spark.sql(f"""select date,product_code,customer_code,sold_quantity from {catalog}.{gold_schema}.sb_orders sbf 
                        inner join incremental_view m on CAST(TRUNC(CAST(sbf.date AS DATE), "MM") AS DATE) = CAST(m.start_month AS DATE)""")
monthly_table.show(10)

In [0]:
spark.sql("select CAST(start_month AS DATE) from incremental_view").show()

In [0]:
spark.sql(f"""
    SELECT DISTINCT CAST(TRUNC(CAST(date AS DATE), "MM") AS DATE) AS gold_months 
    FROM {catalog}.{gold_schema}.sb_orders
""").show()


In [0]:
%sql
select * from fmcg.gold.sb_orders order by date desc limit 2

In [0]:
if(monthly_table.count()==0):
    monthly_table=spark.sql(f"select order_placement_date as date,product_code,customer_id as customer_code,order_qty as sold_quantity  from {catalog}.{silver_schema}.staging_orders")

df_monthly_recalc=monthly_table.withColumn("month_start",F.trunc("date","MM")).groupBy("month_start","product_code","customer_code").agg(F.sum("sold_quantity").alias("sold_quantity")).withColumnRenamed("month_start","date")


df_monthly_recalc.count()


In [0]:
gold_parent=DeltaTable.forName(spark,f"{catalog}.{gold_schema}.fact_{data_source}")
gold_parent.alias("p").merge(source=df_monthly_recalc.alias("c"),condition="c.date = p.date AND p.product_code=c.product_code AND p.customer_code=c.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

#Cleanup

In [0]:
%sql
drop table fmcg.bronze.staging_orders;

In [0]:
%sql
drop table fmcg.silver.staging_orders;